# ✍️ Audience-Tailored Writer: Adverts and Reports from One Source

## Learning Objectives
In this notebook, you will learn:
1. **One source, several audiences** - Generate different documents from the same fact sheet or paper by changing only the instructions
2. **Iterating on a prompt** - Go from a vague advert prompt to one with a required layout and a word limit
3. **Format checks in code** - Verify layout, table rows, word limits and line limits instead of eyeballing them
4. **Faithfulness checks in code** - Flag every number in the output that does not appear in the source
5. **Several reports at once** - Write three audience-specific reports from one research abstract in parallel

## Where this fits
Prompt templates and parallel chains come from Modules 04–07 (`07_parallelization/03_parallelization_lcel.ipynb`). This notebook applies them to a common business job — repurposing one source for different readers — and adds the checks that make the output safe to publish: generated marketing copy and reports love to add plausible numbers that the source never gave.

## Prerequisites
- LCEL chains and `RunnableParallel` from Modules 05–07 (reading only)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: writing adverts and reports
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📱 Part 1: The Product Fact Sheet

You have the technical fact sheets of one smartphone. Try some iterative prompt engineering and do the following:

1. Generate marketing product description for the smartphone

2. Custom product description which has the following:

```
The description should follow this format:

Product Name: <Name of the smartphone>

Description: <Brief Overview of the features>

Product Specifications:
<Table with key product feature specifications>

The description should focus on the most important features
a customer might look for in a phone including the foldable display screen, processing power, RAM, camera and battery life.

After the description, the table should have the
key specifications of the product. It should have two columns.
The first column should have 'Feature'
and the second column should have 'Specification'
and try to put exact numeric values for features if they exist.
Only put these features in the table - foldable display screen, processing power, RAM, camera and battery life
```

3. Custom product description focusing on specific aspects like display, camera and in less than 60 words

In [ ]:
# ============================================================================
# SOURCE 1: the smartphone's technical fact sheet
# ============================================================================
fact_sheet_mobile = """
PRODUCT NAME
Samsung Galaxy Z Fold4 5G Black

PRODUCT OVERVIEW
Stands out. Stands up. Unfolds.
The Galaxy Z Fold4 does a lot in one hand with its 15.73 cm(6.2-inch) Cover Screen.
Unfolded, the 19.21 cm(7.6-inch) Main Screen lets you really get into the zone.
Pushed-back bezels and the Under Display Camera means there's more screen
and no black dot getting between you and the breathtaking Infinity Flex Display.
Do more than more with Multi View. Whether toggling between texts or catching up
on emails, take full advantage of the expansive Main Screen with Multi View.
PC-like power thanks to Qualcomm Snapdragon 8+ Gen 1 processor in your pocket,
transforms apps optimized with One UI to give you menus and more in a glance
New Taskbar for PC-like multitasking. Wipe out tasks in fewer taps. Add
apps to the Taskbar for quick navigation and bouncing between windows when
you're in the groove.4 And with App Pair, one tap launches up to three apps,
all sharing one super-productive screen
Our toughest Samsung Galaxy foldables ever. From the inside out,
Galaxy Z Fold4 is made with materials that are not only stunning,
but stand up to life's bumps and fumbles. The front and rear panels,
made with exclusive Corning Gorilla Glass Victus+, are ready to resist
sneaky scrapes and scratches. With our toughest aluminum frame made with
Armor Aluminum, this is one durable smartphone.
World’s first water resistant foldable smartphones. Be adventurous, rain
or shine. You don't have to sweat the forecast when you've got one of the
world's first water-resistant foldable smartphones.

PRODUCT SPECS
OS - Android 12.0
RAM - 12 GB
Product Dimensions - 15.5 x 13 x 0.6 cm; 263 Grams
Batteries - 2 Lithium Ion batteries required. (included)
Item model number - SM-F936BZKDINU_5
Wireless communication technologies - Cellular
Connectivity technologies - Bluetooth, Wi-Fi, USB, NFC
GPS - True
Special features - Fast Charging Support, Dual SIM, Wireless Charging, Built-In GPS, Water Resistant
Other display features - Wireless
Device interface - primary - Touchscreen
Resolution - 2176x1812
Other camera features - Rear, Front
Form factor - Foldable Screen
Colour - Phantom Black
Battery Power Rating - 4400
Whats in the box - SIM Tray Ejector, USB Cable
Manufacturer - Samsung India pvt Ltd
Country of Origin - China
Item Weight - 263 g
"""

print(f"✅ Fact sheet: {len(fact_sheet_mobile.split())} words")

---
## 📣 Part 2: Three Adverts, One Prompt Iteration at a Time

The three prompts share a role and the fact sheet; each adds constraints: (1) just "write a brief description", (2) a fixed layout with a two-column specification table limited to five features, (3) a catchy version in at most 60 words with emojis. The three are independent, so they run in parallel.

In [ ]:
# ============================================================================
# THREE ADVERT PROMPTS (from vague to specific), RUN IN PARALLEL
# ============================================================================
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

ADVERT_PROMPTS = {
    "brief": """
Act as a marketing manager.
Your task is to help a marketing team create a
description for a retail website advert of a product based
on a technical fact sheet specifications for a mobile smartphone

Write a brief product description

Technical specifications:
{fact_sheet_mobile}
""",
    "structured": """
Act as a marketing manager.
Your task is to help a marketing team create a
description for a retail website advert of a product based
on a technical fact sheet specifications for a mobile smartphone

The description should follow this format:

Product Name: <Name of the smartphone>

Description: <Brief Overview of the features>

Product Specifications:
<Table with key product feature specifications>

The description should focus on the most important features
a customer might look for in a phone including the foldable display screen, processing power, RAM, camera and battery life.

After the description, the table should have the
key specifications of the product. It should have two columns.
The first column should have 'Feature'
and the second column should have 'Specification'
and try to put exact numeric values for features if they exist.
Only put these features in the table - foldable display screen, processing power, RAM, camera and battery life

Technical specifications:
{fact_sheet_mobile}
""",
    "catchy": """
Act as a marketing manager.
Your task is to help a marketing team create a
description for a retail website advert of a product based
on a technical fact sheet specifications for a mobile smartphone

Write a catchy product description with some emojis,
which uses at most 60 words
and focuses on the most important things about the smartphone
which might matter to users like display and camera

Technical specifications:
{fact_sheet_mobile}
""",
}

adverts_chain = RunnableParallel({
    name: ChatPromptTemplate.from_template(p) | llm | StrOutputParser()
    for name, p in ADVERT_PROMPTS.items()
})
adverts = adverts_chain.invoke({"fact_sheet_mobile": fact_sheet_mobile})

for name, text in adverts.items():
    print(f"=============== {name} ({len(text.split())} words) ===============")
    display(Markdown(text))

---
## ✅ Part 3: Checking Format and Faithfulness

Two kinds of checks, both in code:
- **Format** — does the structured advert have its three headings and a table with exactly the five requested features? Is the catchy one within 60 words, with at least one emoji?
- **Faithfulness** — every number in the advert must come from the fact sheet. The fact sheet gives RAM, screen sizes, resolution and battery rating, but **no camera megapixels**; a table that lists "50 MP" was invented by the model.

In [ ]:
# ============================================================================
# CHECK HELPERS: numbers, table rows, emojis
# ============================================================================
import re

NUMBER = re.compile(r"\d+(?:\.\d+)?")


def numbers_in(text: str) -> set[str]:
    return set(NUMBER.findall(text.replace(",", "")))


LIST_NUMBERING = re.compile(r"(?m)^[\s#>*\-]*\d+[.)]\s")  # "1. ", "## 2) " at line start


def unsupported_numbers(output: str, source: str) -> set[str]:
    """Numbers in the output that never appear in the source.
    List numbering ("1.", "2.") is removed first, and 0/1 are ignored ("1 year", "Gen 1")."""
    output = LIST_NUMBERING.sub("", output)
    return {n for n in numbers_in(output) - numbers_in(source) if float(n) > 1}


def table_rows(markdown: str) -> list[list[str]]:
    """Data rows of the first markdown table (header and --- separator removed)."""
    rows = [[c.strip() for c in line.strip().strip("|").split("|")]
            for line in markdown.splitlines() if line.strip().startswith("|")]
    return [r for r in rows[1:] if not all(set(c) <= set("-: ") for c in r)]


def has_emoji(text: str) -> bool:
    return any(ord(ch) >= 0x1F300 for ch in text)


print("✅ Check helpers ready")

In [ ]:
# ============================================================================
# RUN THE CHECKS ON THE THREE ADVERTS
# ============================================================================
structured, catchy = adverts["structured"], adverts["catchy"]
rows = table_rows(structured)
checks = {
    "structured: has Product Name / Description / Product Specifications":
        all(h in structured for h in ["Product Name", "Description", "Product Specifications"]),
    "structured: table has exactly 5 feature rows": len(rows) == 5,
    "catchy: at most 60 words": len(catchy.split()) <= 60,
    "catchy: contains an emoji": has_emoji(catchy),
}
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

print(f"\nTable rows: {[r[0] for r in rows]}")
for name, text in adverts.items():
    extra = unsupported_numbers(text, fact_sheet_mobile)
    print(f"{'✅' if not extra else '⚠️'} {name}: numbers not in the fact sheet → {sorted(extra) or 'none'}")

A ⚠️ line lists numbers a reviewer must confirm or remove before the advert is published — typically camera megapixels or a charging wattage the fact sheet never mentions. The fix belongs in the prompt ("only use numbers from the fact sheet; write 'not specified' otherwise"), and the check stays in place to prove it worked.

---
## 🔬 Part 4: One Abstract, Three Audience Reports

The second source is a research-paper abstract on the ethics of generative AI in healthcare. The task:

Make the model act as an AI expert and transform the given research paper abstract based on the nature of the audience mentioned below.

- Short summary of maximum 10 lines for a general audience
- Detailed report for a healthcare company. Have bullet points for pros and cons of ethics in Generative AI as mentioned in the paper
- Detailed report for a generative AI company solving healthcare problems. Have bullet points for key points mentioned for Generative AI for text, images and structured data based healthcare

The three reports do not depend on each other, so a `RunnableParallel` writes them at the same time from one system prompt and three instructions.

In [ ]:
# ============================================================================
# SOURCE 2: the research-paper abstract
# ============================================================================
paper_abstract = """
The widespread use of ChatGPT and other emerging technology powered by generative
artificial intelligence (AI) has drawn much attention to potential ethical issues, especially in
high-stakes applications such as healthcare.1–3 However, less clear is how to resolve such
issues beyond following guidelines and regulations that are still under discussion and
development. On the other hand, other types of generative AI have been used to synthesize
images and other types of data for research and practical purposes, which have resolved some
ethical issues and exposed other ethical issues,4,5 but such technology is less often the focus
of ongoing ethical discussions. Here we highlight gaps in current ethical discussions of
generative AI via a systematic scoping review of relevant existing research in healthcare, and
reduce the gaps by proposing an ethics checklist for comprehensive assessment and
transparent documentation of ethical discussions in generative AI development. While the
checklist can be readily integrated into the current peer review and publication system to
enhance generative AI research, it may also be used in broader settings to disclose ethicsrelated considerations in generative AI-powered products (or real-life applications of such
products) to help users establish reasonable trust in their capabilities.

Current ethical discussions on generative AI in healthcare
We conducted a systematic scoping review to analyse current ethical discussions on
generative AI in healthcare. Our search in four major academic research databases for
relevant publications from January 2013 to July 2023 yielded 2859 articles (see Methods for
detailed search strategy and Supplementary Figure S1 for the PRISMA flow diagram), of
which 193 articles were included for analysis based on application data modality (text, image,
or structured data), ethical issues discussed, generative AI involved, and whether generative
AI causes or offers technical solutions for issues raised.

Generative AI for text data-based healthcare
Forty-one of the 193 articles discussed ethical considerations pertaining to generative AI
applications for text data, with 20 articles describing methodological developments or
applications of generative AI and the other 21 articles describing review-type works on this
topic. Although some of these review-type articles used the general term “generative AI”, the
main body and supporting evidence focused on LLMs. Twenty-nine articles had in-depth
discussions on ethical issues, whereas the other 12 articles only briefly touched on some
ethical aspects.
Among the 41 articles, 29 articles focused on discussing ethical issues caused by LLMs (and
specifically by GPT in 16 of the articles), covering a wide range of application scenarios and
considered the application of all 10 ethical principles identified in the review (see Figure 1),
as well as other less discussed concerns such as human-AI interaction, and the rights of
LLMs to be considered as co-authors in scientific papers. One paper only commented briefly
on the need for ethical considerations in LLMs and is summarised in the “Others” category.
Although all ethical principles are equally important, some are discussed more often than
others, e.g., non-maleficence (also referred to in the literature as ‘benevolence’), equity, and
privacy.
Fifteen of the 41 articles aimed to resolve some existing ethical issues (for example,
confidentiality of medical data) by using LLMs and other generative AI (e.g., GAN,
autoencoder or diffusion), such as, to reduce privacy concerns by generating synthetic
medical text, to reduce disparity by providing accessible services and assistance, to detect
health-related misinformation, to generate trusted content, and to improve accountability or
transparency over existing approaches. While most articles focused on either identifying
ethical issues caused by generative AI or proposing generative AI-based solutions, three
articles discussed both to provide a more balanced perspective.

Generative AI for image and structured data-based healthcare
Unlike the diverse application scenarios of generative AI based on text data, for image and
structured data, this use of generative AI focuses on data synthesis and encryption. Hence the
majority of articles discussed the methodological developments of generative AI as giving
rise to a more distinctive and focused set of ethical issues.
5
Notably, of the 98 articles on image data and 58 articles on structured data, more than half
(n=63 for image data and n=33 for structured data) only mentioned ethical considerations as a
brief motivation for methodological developments or as a general discussion point. The rest
included more in-depth discussions or evaluations of ethical issues. Among these 155 articles
(as one article covered multiple modalities), 11 articles were review-type work, where 10
articles reviewed methods that mentioned one or two ethical perspectives, and only one
article24 discussed detailed ethical concerns on generative AI applications.
Resolving privacy issues was the main aim of articles for these two data modalities (n=74 for
image data and n=50 for structured data; see Figure 1), predominantly by generating synthetic
data using GAN. Eight articles on image data and 9 articles on structured data used
generative AI to reduce bias, e.g., by synthesizing data for under-represented subgroups in
existing databases. For both data modalities, we did not see explicit discussions on resolving
autonomy, integrity, or morality issues using generative AI, and for structured data the articles
additionally lacked discussions on trust or transparency.
Only 11 articles for image data selectively discussed some ethical issues that generative AI
can give rise to, without specific discussions regarding autonomy, integrity, or morality. For
structured data, only 4 articles discussed equity, privacy, or data security issues caused by
generative AI. Only two articles on structured data included both the cause and resolving
perspectives by discussing ethical issues that may arise from limitations of methods
proposed, specifically bias induced when synthesizing data in order to resolve privacy issues.
"""

print(f"✅ Abstract: {len(paper_abstract.split())} words")

In [ ]:
# ============================================================================
# THREE AUDIENCE REPORTS IN PARALLEL
# ============================================================================
import time

report_prompt = ChatPromptTemplate.from_messages([
    ("system", "Act as an Artificial Intelligence expert. Transform the research paper abstract "
               "below according to the user's instruction. Use only information from the abstract."
               "\n\nAbstract:\n{abstract}"),
    ("human", "{instruction}"),
])
report_chain = report_prompt | llm | StrOutputParser()

INSTRUCTIONS = {
    "general_audience": "Write a short summary of at most 10 lines for a general audience.",
    "healthcare_company": "Write a detailed report for a healthcare company. Include a section 'Pros' "
                          "and a section 'Cons' of ethics in generative AI as mentioned in the paper, "
                          "each with at most 3 bullet points.",
    "genai_company": "Write a detailed report for a generative AI company solving healthcare problems. "
                     "Include one section each on generative AI for text, for images and for "
                     "structured data based healthcare, with the key points from the paper.",
}

reports_chain = RunnableParallel({
    name: (lambda x, i=instr: {"abstract": x["abstract"], "instruction": i}) | report_chain
    for name, instr in INSTRUCTIONS.items()
})

start = time.perf_counter()
reports = reports_chain.invoke({"abstract": paper_abstract})
print(f"⏱️ 3 reports in {time.perf_counter() - start:.1f} s (written in parallel)\n")

for name, text in reports.items():
    print(f"=============== {name} ===============")
    display(Markdown(text))

In [ ]:
# ============================================================================
# CHECK THE REPORTS: line limit, required sections, numbers from the abstract
# ============================================================================
TOP_LEVEL_ITEM = re.compile(r"^(?:[-*•]|\d+[.)])\s")  # unindented bullet or numbered item


def bullets_under(text: str, heading: str) -> int:
    """Top-level list items between a heading that contains `heading` and the next heading."""
    count, inside = 0, False
    for line in text.splitlines():
        stripped = line.strip()
        is_heading = stripped.startswith("#") or (stripped.startswith("**") and stripped.endswith("**"))
        if is_heading or stripped.rstrip(":").lower() == heading.lower():
            inside = heading.lower() in stripped.lower()
        elif inside and TOP_LEVEL_ITEM.match(line):
            count += 1
    return count


summary_lines = [l for l in reports["general_audience"].splitlines() if l.strip()]
hc, gen = reports["healthcare_company"], reports["genai_company"].lower()
checks = {
    f"general audience: at most 10 lines ({len(summary_lines)})": len(summary_lines) <= 10,
    f"healthcare: Pros has 1-3 bullets ({bullets_under(hc, 'Pros')})": 1 <= bullets_under(hc, "Pros") <= 3,
    f"healthcare: Cons has 1-3 bullets ({bullets_under(hc, 'Cons')})": 1 <= bullets_under(hc, "Cons") <= 3,
    "genai company: covers text, images and structured data":
        all(k in gen for k in ["text", "image", "structured data"]),
}
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")
for name, text in reports.items():
    extra = unsupported_numbers(text, paper_abstract)
    print(f"{'✅' if not extra else '⚠️'} {name}: numbers not in the abstract → {sorted(extra) or 'none'}")

---
## 📝 Summary

In this notebook, we learned:

### 1. One source, several audiences
- The source stays fixed; the instruction decides the reader, the length and the layout

### 2. Iterating on a prompt
- Each added constraint (layout, table, word limit) makes the output more usable — and more checkable

### 3. Format checks in code
- Headings, table rows, word and line limits, bullet counts: cheap string checks that run on every output

### 4. Faithfulness checks in code
- Any number in the output that is not in the source is flagged for review; generated copy invents specifications easily

### 5. Several reports at once
- Independent documents from one source run in parallel with `RunnableParallel`

### Next Steps
- **`09_quiz_generator.ipynb`**: study material from a PDF — summary and multiple-choice questions whose evidence is checked against the document